In [1]:
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm.utils import disp_len

In [2]:
#Pick the right device: MPS for Apple Silicon, CUDA for Nvidia, else CPU
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: mps


In [3]:
model_name = "sarvamai/sarvam-translate"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [4]:
model

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [5]:
def translate(
        target_language:str, 
        input_text:str,
        printall : bool = False 
        ) -> str:
        
    # Chat-style message prompt
    messages = [
        {"role": "system", "content": f"Translate the text below to {target_language}."},
        {"role": "user", "content": input_text}
    ]

    # Apply chat template to structure the conversation
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    display("Chat Template Text", text)

    # Tokenize and move input to model device
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate the output
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.01,
        num_return_sequences=1
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    output_text = tokenizer.decode(output_ids, skip_special_tokens=True)
    
    if printall:
        print("Input:", input_text)
        print(f"{target_language} Translation:", output_text)

    return output_text

In [6]:
# Translation task
target = "Hindi"
input_text = "Be the change you wish to see in the world."

translation =translate(target_language=target,input_text=input_text,printall=True)


'Chat Template Text'

'<bos><start_of_turn>user\nTranslate the text below to Hindi.\n\nBe the change you wish to see in the world.<end_of_turn>\n<start_of_turn>model\n'

Input: Be the change you wish to see in the world.
Hindi Translation: दुनिया में जो बदलाव आप देखना चाहते हैं, वह बदलाव खुद बनें।


In [7]:
# Translation task
target = "Marathi"
input_text = "The error is crystal clear! Your notebook is running on a Mac"

translation = translate(target_language=target,input_text=input_text, printall=True)



'Chat Template Text'

'<bos><start_of_turn>user\nTranslate the text below to Marathi.\n\nThe error is crystal clear! Your notebook is running on a Mac<end_of_turn>\n<start_of_turn>model\n'

Input: The error is crystal clear! Your notebook is running on a Mac
Marathi Translation: चूक अगदी स्पष्ट आहे! तुमची नोटबुक मॅकवर चालतेय.


In [8]:
# Translation task
target = "English"
input_text = "चूक अगदी स्पष्ट आहे! तुमची नोटबुक मॅकवर चालतेय."

translation = translate(target_language=target,input_text=input_text, printall=True)



'Chat Template Text'

'<bos><start_of_turn>user\nTranslate the text below to English.\n\nचूक अगदी स्पष्ट आहे! तुमची नोटबुक मॅकवर चालतेय.<end_of_turn>\n<start_of_turn>model\n'

Input: चूक अगदी स्पष्ट आहे! तुमची नोटबुक मॅकवर चालतेय.
English Translation: The error is  Your notebook is running on a Mac.


In [9]:
# Translation task
target = "English"
input_text = "दुनिया में जो बदलाव आप देखना चाहते हैं, वह बदलाव खुद बनें।"

translation = translate(target_language=target,input_text=input_text, printall=True)
##Expected = "Be the change you wish to see in the world."



'Chat Template Text'

'<bos><start_of_turn>user\nTranslate the text below to English.\n\nदुनिया में जो बदलाव आप देखना चाहते हैं, वह बदलाव खुद बनें।<end_of_turn>\n<start_of_turn>model\n'

Input: दुनिया में जो बदलाव आप देखना चाहते हैं, वह बदलाव खुद बनें।
English Translation: Be the change you wish to see in the world.
